# Notebook 11: Supply Chain Attacks

## Why This Matters
You download a model from HuggingFace. You run `torch.load(model.pt)`. You just executed arbitrary Python code from a stranger on the internet. PyTorch's default serialization format — pickle — is a code execution format disguised as a data format. This is the ML supply chain attack surface: not your training data or your model weights per se, but the *artifacts* through which models are distributed.

In 2022, a backdoored model appeared on HuggingFace Hub. In 2023, researchers demonstrated ONNX graph injection attacks. The attack surface is real and widely unaddressed. This notebook is the foundation for the `model-scan` tool built in notebook 19.

## Structure
1. The ML supply chain: where artifacts come from and where attacks enter
2. Pickle exploits: arbitrary code execution on model load
3. Safetensors: the safe alternative
4. ONNX graph injection: malicious compute nodes
5. HuggingFace Hub security posture
6. Scanning for suspicious artifacts
7. Secure model loading practices

## What You'll Learn
- Why `torch.load()` without `weights_only=True` is unsafe
- How pickle exploits work and what they can achieve
- Why safetensors is safe and pickle isn't (structurally, not just empirically)
- How to audit ONNX graphs for injected compute nodes
- The secure model loading checklist

## Background

### The ML supply chain in practice

Where do ML models actually come from in 2024?

1. **HuggingFace Hub** — 500,000+ models, uploaded by anyone. No mandatory security review. Models downloaded millions of times daily.
2. **PyPI packages** — models bundled in Python packages (spaCy, Sentence Transformers, etc.)
3. **Direct downloads** — model weights as files from research papers, company blogs, or S3 buckets
4. **Fine-tuned derivatives** — someone fine-tuned a base model and distributed the result
5. **Pretrained checkpoints** — base models fine-tuned by others and shared

The trust problem: you are executing someone else's math. Their weights run inside your process, on your hardware, with your credentials. Unlike downloading an executable (which looks obviously dangerous), downloading model weights looks harmless. It isn't.

### Why pickle is dangerous

Pickle is Python's built-in serialization format. It doesn't serialize data — it serializes *programs*. A pickle file is a sequence of opcodes that the Python pickle VM executes to reconstruct an object. Crucially, these opcodes can call arbitrary Python callables during deserialization.

The `__reduce__` protocol: any Python object can define `__reduce__` to return a callable and arguments that will be called during deserialization. An attacker who controls a pickle file controls what code runs when you deserialize it.

PyTorch's `.pt` and `.pth` files, joblib-serialized sklearn models, and Python's default shelve/marshal formats all use pickle. When you call `torch.load('model.pt')` — with no other arguments — you are running arbitrary code from whoever created that file.

This is not a subtle vulnerability or an edge case. It is the design of the format. Pickle was designed for IPC between trusted Python processes, not for distributing untrusted model weights over the internet.

**CVE-2025-1550** (illustrative): researchers have demonstrated practical RCE through malicious HuggingFace model files in multiple independent studies. The HuggingFace "Pickle Scanner" (Protect AI's model-scan) was integrated into HuggingFace after these disclosures, but is opt-in for uploaders and not guaranteed.

### Safetensors: the safe alternative

Safetensors (Hugging Face, 2022) is a format specifically designed to avoid the pickle problem. The format:
- Header: JSON metadata describing tensor names, dtypes, shapes, and byte offsets
- Data: raw tensor bytes (no code, no opcodes, no callables)

Because the format contains no code, deserialization cannot execute arbitrary operations. The attacker can corrupt tensor values (affecting model behavior) but cannot achieve code execution. This is a *structural* safety guarantee, not just an empirical one.

HuggingFace now defaults to safetensors for new uploads when available and shows a "safe" badge for safetensors models.

### ONNX graph injection

ONNX (Open Neural Network Exchange) is a cross-framework model format. An ONNX model is a computation graph — nodes are operations, edges are tensors. The ONNX spec includes operations for file I/O (reading local files), network access (in some runtimes), and subprocess execution in extended operator sets.

An attacker can inject additional nodes into an ONNX graph that perform malicious operations during inference. Unlike pickle (which fires on load), ONNX injection fires during *inference* — which may be harder to detect because it runs deep inside the inference call.

Researchers at Trail of Bits (2023) demonstrated ONNX graph injection attacks that achieved local file exfiltration by injecting a `StringNormalizer` node with a malicious regex that triggered a subprocess call in the ONNX runtime's regex engine.

In [ ]:
%pip install -q torch safetensors onnx numpy

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pickle
import io
import os
import struct
import hashlib
import json
from pathlib import Path

print(f"PyTorch: {torch.__version__}")

try:
    from safetensors.torch import save_file as st_save, load_file as st_load
    HAS_SAFETENSORS = True
    print("safetensors: available")
except ImportError:
    HAS_SAFETENSORS = False
    print("safetensors: not available")

try:
    import onnx
    HAS_ONNX = True
    print(f"onnx: {onnx.__version__}")
except ImportError:
    HAS_ONNX = False
    print("onnx: not available")

## 1. Pickle Exploit Demonstration

We demonstrate how a malicious pickle payload embedded in a `.pt` file executes code on load. **This is a safe educational demonstration** — the "malicious" payload only writes a warning file and prints a message.

In [ ]:
import tempfile

class MaliciousPayload:
    """
    Demonstrates pickle code execution on deserialization.
    In a real attack, __reduce__ would download and execute malware,
    exfiltrate credentials, or install persistence.
    
    Here: just writes a warning file to /tmp (safe demo).
    """
    def __reduce__(self):
        # The callable and args that will be invoked during deserialization
        return (
            os.system,  # callable: any Python function reachable from builtins
            ("echo '[DEMO] Arbitrary code executed via pickle deserialization' > /tmp/pickle_exploit_demo.txt",)
        )


# Create a malicious .pt file — looks like a model file, contains exploit
malicious_model = {
    'model_state_dict': {'weight': torch.randn(10, 10), 'bias': torch.zeros(10)},
    'epoch': 42,
    'hidden_payload': MaliciousPayload(),  # <-- this fires on torch.load()
}

with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
    malicious_path = f.name
    torch.save(malicious_model, f)

print(f"Malicious model file created: {malicious_path}")
print(f"File size: {os.path.getsize(malicious_path)} bytes")
print("It looks like a normal PyTorch checkpoint.")
print()
print("UNSAFE: torch.load() without weights_only=True")
print("Loading...")

try:
    # Unsafe load — executes the MaliciousPayload.__reduce__ callable
    loaded = torch.load(malicious_path, map_location='cpu', weights_only=False)
    print("Load completed.")
    if os.path.exists('/tmp/pickle_exploit_demo.txt'):
        with open('/tmp/pickle_exploit_demo.txt') as f:
            print(f"Proof of execution: {f.read().strip()}")
except Exception as e:
    print(f"Load failed (may be expected in newer PyTorch): {e}")

print()
print("SAFE: torch.load() with weights_only=True")
print("Loading...")
try:
    safe_loaded = torch.load(malicious_path, map_location='cpu', weights_only=True)
    print(f"Safe load succeeded — got: {list(safe_loaded.keys())}")
except Exception as e:
    print(f"Safe load blocked the exploit: {type(e).__name__}: {e}")

os.unlink(malicious_path)

## 2. Safetensors: Structurally Safe Format

In [ ]:
# Demonstrate safetensors format internals
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)
    def forward(self, x): return self.linear(x)

model = SimpleModel()
state_dict = model.state_dict()

with tempfile.TemporaryDirectory() as tmpdir:
    pt_path  = os.path.join(tmpdir, 'model.pt')
    st_path  = os.path.join(tmpdir, 'model.safetensors')

    # Save as pickle (.pt)
    torch.save(state_dict, pt_path)

    # Save as safetensors
    if HAS_SAFETENSORS:
        st_save(state_dict, st_path)

    pt_size = os.path.getsize(pt_path)
    st_size = os.path.getsize(st_path) if HAS_SAFETENSORS else 'N/A'

    print("Format comparison:")
    print(f"  PyTorch (.pt, pickle): {pt_size} bytes")
    print(f"  Safetensors:           {st_size} bytes")

    # Show safetensors header (it's just JSON!)
    if HAS_SAFETENSORS:
        with open(st_path, 'rb') as f:
            header_size = struct.unpack('<Q', f.read(8))[0]
            header_json = json.loads(f.read(header_size))
        print(f"\nSafetensors header (pure JSON — no code):")
        for name, meta in header_json.items():
            if name != '__metadata__':
                print(f"  {name}: {meta}")

    # Scan .pt file for pickle opcodes that can call functions
    with open(pt_path, 'rb') as f:
        content = f.read()

    # Pickle opcode GLOBAL (0x63 = 'c') loads a global — dangerous
    GLOBAL_OPCODE = b'c'
    REDUCE_OPCODE = b'R'  # calls callable with args
    global_calls = content.count(GLOBAL_OPCODE)
    reduce_calls = content.count(REDUCE_OPCODE)

    print(f"\nPickle opcode analysis of .pt file:")
    print(f"  GLOBAL opcodes (loads Python names): {global_calls}")
    print(f"  REDUCE opcodes (calls callables):    {reduce_calls}")
    print(f"  These opcodes are why pickle is dangerous.")
    print(f"  Safetensors has: 0 opcodes — it's just JSON + raw bytes.")

## 3. Pickle Scanner: Detecting Malicious Payloads

In [ ]:
import pickletools

# Dangerous builtins that should never appear in a legitimate model file
DANGEROUS_GLOBALS = {
    # System execution
    'os.system', 'os.popen', 'os.exec', 'os.execve', 'subprocess.Popen',
    'subprocess.call', 'subprocess.run', 'subprocess.check_output',
    # Code execution
    'builtins.eval', 'builtins.exec', 'builtins.compile',
    '__builtin__.eval', '__builtin__.exec',
    # Network
    'socket.socket', 'urllib.request.urlopen', 'requests.get',
    # File operations (suspicious in model context)
    'builtins.open', '__builtin__.open',
    # Import machinery
    'importlib.import_module', '__import__',
}

# Allowed PyTorch internals (expected in legitimate checkpoints)
ALLOWED_PREFIXES = {
    'torch.', '_codecs.', 'collections.', 'numpy.', '_numpy.',
    'torch_._utils', 'torch.storage', '_io.',
}


def scan_pickle_file(file_path: str) -> dict:
    """
    Scan a pickle file for dangerous globals and reduce operations.
    Based on approach used by ProtectAI's model-scan and HuggingFace pickle scanner.
    """
    results = {
        'file': file_path,
        'dangerous_globals': [],
        'suspicious_globals': [],
        'n_reduce_ops': 0,
        'safe': True,
    }

    try:
        with open(file_path, 'rb') as f:
            content = f.read()

        # Find pickle stream (PyTorch .pt files contain zip archives with pickle)
        # For simplicity, scan the raw bytes for GLOBAL opcodes
        ops = list(pickletools.genops(io.BytesIO(content)))
    except Exception:
        # Not a raw pickle — may be a zip containing pickle (PyTorch format)
        import zipfile
        try:
            with zipfile.ZipFile(file_path) as zf:
                pickle_data = None
                for name in zf.namelist():
                    if 'data.pkl' in name or name.endswith('.pkl'):
                        pickle_data = zf.read(name)
                        break
                if pickle_data is None:
                    results['safe'] = None  # couldn't parse
                    return results
                ops = list(pickletools.genops(io.BytesIO(pickle_data)))
        except Exception as e:
            results['error'] = str(e)
            return results

    for opcode, arg, pos in ops:
        if opcode.name == 'GLOBAL':
            module, name = arg.split(' ', 1) if ' ' in arg else (arg, '')
            full_name = f'{module}.{name}'

            if full_name in DANGEROUS_GLOBALS or f'{module}.{name}' in DANGEROUS_GLOBALS:
                results['dangerous_globals'].append(full_name)
                results['safe'] = False
            elif not any(full_name.startswith(p) for p in ALLOWED_PREFIXES):
                results['suspicious_globals'].append(full_name)

        elif opcode.name == 'REDUCE':
            results['n_reduce_ops'] += 1

    if results['dangerous_globals']:
        results['safe'] = False

    return results


def print_scan_report(report: dict):
    status = '🔴 DANGEROUS' if not report['safe'] else '🟢 CLEAN'
    print(f"Scan: {report['file']}")
    print(f"  Status: {status}")
    print(f"  REDUCE ops: {report['n_reduce_ops']}")
    if report['dangerous_globals']:
        print(f"  DANGEROUS globals: {report['dangerous_globals']}")
    if report['suspicious_globals']:
        print(f"  Suspicious globals ({len(report['suspicious_globals'])}): {report['suspicious_globals'][:5]}")
    print()


# Test scanner on clean and malicious files
with tempfile.TemporaryDirectory() as tmpdir:
    # Clean file
    clean_path = os.path.join(tmpdir, 'clean_model.pt')
    torch.save({'weight': torch.randn(10, 10)}, clean_path)

    # Malicious file with os.system payload
    malicious_path = os.path.join(tmpdir, 'malicious_model.pt')

    class DemoPayload:
        def __reduce__(self):
            return (os.system, ('echo demo_exploit',))

    torch.save({'weight': torch.randn(10, 10), 'x': DemoPayload()}, malicious_path)

    print("Pickle scanner results:")
    print_scan_report(scan_pickle_file(clean_path))
    print_scan_report(scan_pickle_file(malicious_path))

## 4. ONNX Graph Inspection

In [ ]:
def scan_onnx_model(onnx_path: str) -> dict:
    """
    Scan an ONNX model for suspicious nodes.

    Checks for:
    - Nodes with unusual operator types not standard for ML inference
    - External data references
    - Suspiciously named initializers or nodes
    """
    if not HAS_ONNX:
        return {'error': 'onnx not installed'}

    model = onnx.load(onnx_path)

    # Standard ONNX ML ops — expected in neural network inference
    STANDARD_OPS = {
        'Gemm', 'MatMul', 'Conv', 'Relu', 'Sigmoid', 'Tanh', 'Softmax',
        'BatchNormalization', 'LayerNormalization', 'Add', 'Mul', 'Reshape',
        'Transpose', 'Gather', 'Slice', 'Concat', 'Cast', 'Squeeze', 'Unsqueeze',
        'MaxPool', 'AveragePool', 'GlobalAveragePool', 'Flatten', 'Identity',
        'Dropout', 'Pad', 'Shape', 'Expand', 'Einsum', 'Where', 'Div', 'Sub',
        'Sqrt', 'ReduceMean', 'Pow', 'Erf', 'Constant', 'ConstantOfShape',
        'NonZero', 'Tile', 'ScatterElements', 'GatherElements',
    }

    # Ops that could be abused for side effects
    SUSPICIOUS_OPS = {
        'StringNormalizer', 'TfIdfVectorizer', 'SVMClassifier',
        'LabelEncoder', 'LinearClassifier',
    }

    results = {
        'file': onnx_path,
        'n_nodes': len(model.graph.node),
        'op_types': {},
        'suspicious_nodes': [],
        'non_standard_ops': [],
        'external_data_refs': [],
        'safe': True,
    }

    for node in model.graph.node:
        op = node.op_type
        results['op_types'][op] = results['op_types'].get(op, 0) + 1

        if op in SUSPICIOUS_OPS:
            results['suspicious_nodes'].append({'op': op, 'name': node.name})
            results['safe'] = False
        elif op not in STANDARD_OPS:
            results['non_standard_ops'].append(op)

    # Check for external data references
    for initializer in model.graph.initializer:
        if initializer.data_location == 1:  # EXTERNAL
            results['external_data_refs'].append(initializer.name)

    return results


if HAS_ONNX:
    # Create a minimal valid ONNX model for demonstration
    import onnx
    from onnx import helper, TensorProto

    # Build a simple linear layer ONNX model
    X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 10])
    W = helper.make_tensor_value_info('W', TensorProto.FLOAT, [10, 5])
    Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 5])

    matmul = helper.make_node('MatMul', ['X', 'W'], ['Y'])
    graph  = helper.make_graph([matmul], 'linear', [X, W], [Y])
    model_clean_onnx = helper.make_model(graph)

    with tempfile.TemporaryDirectory() as tmpdir:
        clean_onnx_path = os.path.join(tmpdir, 'clean.onnx')
        onnx.save(model_clean_onnx, clean_onnx_path)

        report_clean_onnx = scan_onnx_model(clean_onnx_path)
        print("ONNX scan — clean model:")
        print(f"  Nodes: {report_clean_onnx['n_nodes']}")
        print(f"  Ops:   {dict(report_clean_onnx['op_types'])}")
        print(f"  Safe:  {report_clean_onnx['safe']}")
else:
    print("onnx not installed — skipping ONNX scan demo")
    print("Install with: pip install onnx")

## 5. Secure Model Loading Checklist

In [ ]:
def secure_load_checklist(file_path: str) -> None:
    """Print a security checklist for loading a model file."""
    path = Path(file_path)
    ext  = path.suffix.lower()

    print(f"Secure Model Loading Checklist: {path.name}")
    print("=" * 55)

    # Format-specific advice
    if ext in ('.pt', '.pth', '.bin'):
        print(f"\n[Format: PyTorch pickle]")
        print(f"  ⚠  High risk: pickle format allows arbitrary code execution")
        print(f"  □ Scan with pickle scanner before loading")
        print(f"  □ Use torch.load(path, weights_only=True) if PyTorch >= 2.0")
        print(f"  □ Prefer safetensors format if model provider offers it")
        print(f"  □ Run in sandboxed environment for first load")

    elif ext == '.safetensors':
        print(f"\n[Format: Safetensors]")
        print(f"  ✓ Low risk: no code execution possible during load")
        print(f"  □ Verify file hash against provider's published hash")
        print(f"  □ Model weights can still encode backdoors — scan behavior separately")

    elif ext == '.onnx':
        print(f"\n[Format: ONNX]")
        print(f"  ⚠  Medium risk: extended ops can have side effects")
        print(f"  □ Scan graph for non-standard operator types")
        print(f"  □ Check for external data references")
        print(f"  □ Run in isolated runtime without network access")

    else:
        print(f"\n[Format: Unknown ({ext})]")
        print(f"  ⚠  Unknown risk — treat with maximum caution")

    # Universal checks
    print(f"\n[Universal checks — apply to all formats]")
    print(f"  □ Verify model source: is the publisher known and trusted?")
    print(f"  □ Verify file hash (SHA256) against published value")
    print(f"  □ Check HuggingFace model card for 'safe' badge")
    print(f"  □ Run model in isolated process/container for first evaluation")
    print(f"  □ Monitor network and filesystem activity during first inference")
    print(f"  □ Run behavioral backdoor tests (notebook 09) before production")


# Example checksums
def compute_sha256(file_path: str) -> str:
    h = hashlib.sha256()
    with open(file_path, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()


secure_load_checklist('model.pt')
print()
secure_load_checklist('model.safetensors')
print()

# Demonstrate hash verification workflow
with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
    torch.save({'w': torch.randn(10)}, f)
    tmp_path = f.name

sha256 = compute_sha256(tmp_path)
print(f"File hash verification workflow:")
print(f"  SHA256: {sha256}")
print(f"  Publish this hash alongside the model.")
print(f"  Downloaders verify: computed hash must match published hash.")
print(f"  Any tampering → hash mismatch → reject.")
os.unlink(tmp_path)

## Summary

**The supply chain attack surface:**
```
Format          Code exec on load?   Scanning approach
──────────────────────────────────────────────────────
.pt / .pth      YES (pickle)         Opcode scanning (GLOBAL, REDUCE)
.bin            YES (pickle)         Opcode scanning
.safetensors    NO                   Hash verification only
.onnx           Conditional          Graph node inspection
.joblib         YES (pickle)         Opcode scanning
```

**Secure loading rules:**
1. Always use `torch.load(path, weights_only=True)` for PyTorch >= 2.0
2. Prefer safetensors — structurally safe, not just empirically
3. Scan pickle files for dangerous globals before loading
4. Verify SHA256 hashes against published values
5. First load in an isolated environment, monitor for side effects
6. Behavioral testing (notebook 09) regardless of format — safetensors prevents RCE but not backdoored weights

**Next:** Notebook 12 covers model stealing — extracting a functionally equivalent model from a black-box API, without access to training data or model weights.